# SQL Project: E-commerce Analytics System


## Problem statement

You are a data engineer for an e-commerce company.

**Deliver:**
- Database schema
- Seed data
- Analytics queries

**Goal:** business insights from SQL.


## Prerequisites

- PostgreSQL running ([`README.md`](README.md)).
- This project **drops** `orders`, `products`, `users` if they exist (same names as earlier lessons).


## Schema design


In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="de_course",
    user="admin",
    password="admin",
)

cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders CASCADE")
cur.execute("DROP TABLE IF EXISTS products CASCADE")
cur.execute("DROP TABLE IF EXISTS users CASCADE")

# USERS
cur.execute(
    """
CREATE TABLE users (
    id SERIAL PRIMARY KEY,
    name TEXT,
    city TEXT
)
"""
)

# PRODUCTS
cur.execute(
    """
CREATE TABLE products (
    id SERIAL PRIMARY KEY,
    name TEXT,
    category TEXT,
    price INT
)
"""
)

# ORDERS
cur.execute(
    """
CREATE TABLE orders (
    id SERIAL PRIMARY KEY,
    user_id INT,
    product_id INT,
    quantity INT,
    order_date DATE
)
"""
)

conn.commit()
print("Schema ready: users, products, orders.")


## Insert sample data


In [ ]:
# USERS
cur.execute(
    """
INSERT INTO users (name, city)
VALUES
('Amar', 'Bangalore'),
('Neha', 'Delhi'),
('Ravi', 'Mumbai')
"""
)

# PRODUCTS
cur.execute(
    """
INSERT INTO products (name, category, price)
VALUES
('Phone', 'Electronics', 1000),
('Laptop', 'Electronics', 2000),
('Shoes', 'Fashion', 500)
"""
)

# ORDERS
cur.execute(
    """
INSERT INTO orders (user_id, product_id, quantity, order_date)
VALUES
(1, 1, 1, '2024-01-01'),
(1, 2, 1, '2024-01-02'),
(2, 3, 2, '2024-01-03'),
(3, 1, 1, '2024-01-04')
"""
)

conn.commit()
print("Seed data loaded.")


## Business queries

### 1) Total revenue


In [ ]:
cur.execute(
    """
SELECT SUM(products.price * orders.quantity) AS total_revenue
FROM orders
JOIN products ON products.id = orders.product_id
"""
)

print(cur.fetchall())


### 2) Revenue per user


In [ ]:
cur.execute(
    """
SELECT users.name,
SUM(products.price * orders.quantity) AS revenue
FROM orders
JOIN users ON users.id = orders.user_id
JOIN products ON products.id = orders.product_id
GROUP BY users.name
"""
)

print(cur.fetchall())


### 3) Top-selling products


In [ ]:
cur.execute(
    """
SELECT products.name,
SUM(orders.quantity) AS total_sold
FROM orders
JOIN products ON products.id = orders.product_id
GROUP BY products.name
ORDER BY total_sold DESC
"""
)

print(cur.fetchall())


### 4) Revenue per category


In [ ]:
cur.execute(
    """
SELECT products.category,
SUM(products.price * orders.quantity) AS revenue
FROM orders
JOIN products ON products.id = orders.product_id
GROUP BY products.category
"""
)

print(cur.fetchall())


## Advanced queries

### 5) Top customer (window function)

_PostgreSQL:_ aggregate first in a subquery, then `RANK()` on `total_spent` (cannot nest `SUM` inside `OVER` with `GROUP BY` in one select).


In [ ]:
cur.execute(
    """
SELECT *
FROM (
    SELECT name,
           total_spent,
           RANK() OVER (ORDER BY total_spent DESC) AS rank
    FROM (
        SELECT users.name,
               SUM(products.price * orders.quantity) AS total_spent
        FROM orders
        JOIN users ON users.id = orders.user_id
        JOIN products ON products.id = orders.product_id
        GROUP BY users.name
    ) AS per_user
) AS ranked
WHERE rank = 1
"""
)

print(cur.fetchall())


### 6) Daily revenue


In [ ]:
cur.execute(
    """
SELECT order_date,
SUM(products.price * orders.quantity) AS revenue
FROM orders
JOIN products ON products.id = orders.product_id
GROUP BY order_date
ORDER BY order_date
"""
)

print(cur.fetchall())


## Practice

1. Find:
   - Most active user (by order count)
   - Average order value
   - Orders per city


## Assignment

Extend the system:

1. Add **`payments`** table
2. Add **order status** (`completed`, `pending`, …)

Then answer:
- Revenue by city
- Failed vs completed orders (define “failed” in your schema)
- Monthly revenue trend

**Bonus:** window functions for ranking (e.g. users by revenue).


## Interview Questions

1. How do you design an analytics system?
2. How do you calculate revenue?
3. How do you optimize joins?
4. How would you scale this system?


## What you just built

- End-to-end **schema + seed + analytics**
- **Joins**, **aggregations**, **windows**

👉 Same shape as real warehouse-facing work—smaller data.


---

### Phase 2 wrap-up

You can tell a credible SQL story in interviews: modeling, metrics, and tuning intuition.

**Next — Phase 3:** **ETL pipelines** — Python + SQL + data flow end-to-end.


## Optional: close connection


In [ ]:
cur.close()
conn.close()
print("Connection closed.")


## Your tasks

- [ ] Run **Schema** → **Daily revenue** with Postgres up.
- [ ] **Practice:** most active user; AOV = total revenue ÷ order count; orders grouped by `users.city`.
- [ ] **Assignment:** `payments` + `status` on orders; revenue by city; completed vs failed; monthly trend (`DATE_TRUNC('month', order_date)`); **bonus:** rank users.
- [ ] Add **2–3 custom** business questions (COGS if you add cost, repeat buyers, etc.).
- [ ] Draft short answers to interview questions (scale → partitioning, read replicas, OLTP vs OLAP).
